## DataFoundry Entity: history tab (interface)
### GOAL: Get temperature dataset for the last 3-5 days and upload to DataFoundry entity dataset

In [5]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

## Import weather forecast data

In [7]:
# api-endpoint
URL = "https://pro.openweathermap.org/data/2.5/forecast/hourly?lat=51.3135296&lon=4.83753463479851&appid=273a7bdc5f26d184400ce98d1bc8e957&units=metric"
 
# Sending a GET request to the specified URL to retrieve weather data
response = requests.get(url = URL)

# Extracting data from the response in JSON format
data = response.json()

# Flatten (normalize) the JSON file to create a DataFrame from the 'list' key
list = pd.json_normalize(data, record_path=['list'])  # Extract main data (list)
weather = pd.json_normalize(data, record_path=['list', 'weather'])  # Extract weather descriptions into a separate DataFrame

# Combining the main data and weather DataFrame, and dropping unnecessary columns
forecast = pd.concat([list, weather], axis=1)
forecast = forecast.drop(["visibility", "weather", 'description', 'main', "dt", "id", "pop", "main.feels_like", "main.pressure", "clouds.all", "wind.speed", "wind.deg", "wind.gust", "sys.pod", "rain.1h", "id", "icon", "main.sea_level", "main.grnd_level", "main.humidity", "main.temp_kf"], axis='columns')
forecast

,dt_txt,main.temp,main.temp_min,main.temp_max
0,2024-10-04 08:00:00,6.73,6.73,9.76
1,2024-10-04 09:00:00,8.23,8.23,11.63
2,2024-10-04 10:00:00,10.30,10.30,13.19
3,2024-10-04 11:00:00,12.58,12.58,14.23
4,2024-10-04 12:00:00,14.89,14.89,14.89
...,...,...,...,...
91,2024-10-08 03:00:00,11.33,11.33,11.33
92,2024-10-08 04:00:00,11.33,11.33,11.33
93,2024-10-08 05:00:00,11.04,11.04,11.04
94,2024-10-08 06:00:00,10.76,10.76,10.76


In [20]:
#json
import requests

# api-endpoint
URL = "https://pro.openweathermap.org/data/2.5/forecast/hourly?lat=51.3135296&lon=4.83753463479851&appid=273a7bdc5f26d184400ce98d1bc8e957&units=metric"
 
# Sending a GET request to the specified URL to retrieve weather data
response = requests.get(url = URL)

# Extracting data from the response in JSON format
data = response.json()

# Flatten (normalize) the JSON file to create a DataFrame from the 'list' key
list = pd.json_normalize(data, record_path=['list'])  # Extract main data (list)
weather = pd.json_normalize(data, record_path=['list', 'weather'])  # Extract weather descriptions into a separate DataFrame

# Combining the main data and weather DataFrame, and dropping unnecessary columns
forecast = pd.concat([list, weather], axis=1)
forecast = forecast.drop(["main.temp_min", "main.temp_max", "visibility", 'description', "weather", 'main', "dt", "id", "pop", "main.feels_like", "main.pressure", "clouds.all", "wind.speed", "wind.deg", "wind.gust", "sys.pod", "rain.1h", "id", "icon", "main.sea_level", "main.grnd_level", "main.humidity", "main.temp_kf"], axis='columns')

# Renaming columns for better readability and consistency
forecast = forecast.rename(columns={"dt_txt": "ts"}, errors="raise")
forecast = forecast.rename(columns={"main.temp": "Temperature_API"}, errors="raise")

# Converting the 'ts' column to datetime format
forecast['ts'] = pd.to_datetime(forecast['ts'])  # Turn timestamp into datetime dtype

# Getting the current time and adding one hour to it
start_time = datetime.now() + timedelta(hours=0)  # Set the start time to the next hour from the current time

# Filtering the forecast DataFrame to include only rows where the timestamp is greater than or equal to the start time
forecast = forecast[forecast['ts'] >= start_time]  # Keep only future forecasts starting from one hour from now

# Rounding temperature values to the nearest whole number
forecast = forecast.round(0)  # Round temperature to 0 decimal places

# Extracting hour and date from the 'ts' column for further analysis
hours = forecast['ts'].dt.hour
date = forecast['ts'].dt.date
forecast['hr'] = hours  # Adding hour to the DataFrame
forecast['date'] = date  # Adding date to the DataFrame
forecast = forecast.drop(['date'], axis='columns')  # Dropping the 'date' column as it's no longer needed

# Filtering the forecast DataFrame to exclude rows where hour is between 22 and 23 or between 0 and 7
# filtered_forecast = forecast[(forecast['hr'] < 23) & (forecast['hr'] > 6)]

# Selecting the first 12 rows of the filtered forecast DataFrame for display
forecast

,ts,Temperature_API,hr
2,2024-10-04 10:00:00,10.0,10
3,2024-10-04 11:00:00,13.0,11
4,2024-10-04 12:00:00,15.0,12
5,2024-10-04 13:00:00,15.0,13
6,2024-10-04 14:00:00,15.0,14
...,...,...,...
91,2024-10-08 03:00:00,11.0,3
92,2024-10-08 04:00:00,11.0,4
93,2024-10-08 05:00:00,11.0,5
94,2024-10-08 06:00:00,11.0,6


### Import csv files from DataFoundry

In [9]:
# Get the database from the DataFoundry link
df_indoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/Yy9PQlp3clNidmNyc0pYS1JBV1NlQ1JpbGNKWHBIQlVVMjlwQW9nOFY5UT0=", low_memory=False)
df_indoor = df_indoor[(df_indoor.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_indoor = df_indoor.drop(["Unnamed: 7", "device_id", "activity", "participant", "sender", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_indoor = df_indoor.rename(columns={"Temperature": "Temperature_indoor"}, errors="raise")
df_indoor['ts'] = pd.to_datetime(df_indoor['ts']) ## Turn timestamp into datetime dtype
df_indoor['Temperature_indoor'] = np.round(df_indoor['Temperature_indoor'] * 10) / 10 ## round temperature to 1 decimal
df_indoor = df_indoor.dropna() ## drop rows with empty (NA) cells
df_indoor = df_indoor.drop_duplicates() ## Drop duplicate rowsindex_list= df_indoor2.Timestamp[(df_indoor2.Timestamp >= "2024-08-08 16:00:00") & (df_indoor2.Timestamp <= "2024-08-08 19:20:00")].index.tolist(
df_indoor['hr'] = df_indoor['ts'].dt.hour
df_indoor['date'] = df_indoor['ts'].dt.date
df_indoor = df_indoor.groupby(['date', 'hr']).first().reset_index()
df_indoor['ts'] = df_indoor["ts"].dt.round('h')  #Round the datestamp column to hours
df_indoor = df_indoor.drop(["hr", "date"], axis='columns')

In [13]:
# Get the database from the DataFoundry link
df_API = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/N0FsN2loZlVYMWhBaE0rd0l5T2NadzR3YTNBcnovQlJ0SG13dHMxL0U1RT0=")
df_API = df_API[(df_API.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_API = df_API.drop(["activity", "device_id", "sender", "participant", "Unnamed: 7", "id", "recipient", "pp1", "pp2", "pp3", "Temperature_API_MAX", "Temperature_API_MIN", "weather_description", "weather_main"], axis='columns')
df_API['ts'] = pd.to_datetime(df_API['ts']) ## Turn timestamp into datetime dtype
df_API['Temperature_API'] = np.round(df_API['Temperature_API'] * 10) / 10 ## round temperature to 1 decimal
df_API['hr'] = df_API['ts'].dt.hour
df_API['date'] = df_API['ts'].dt.date
df_API = df_API.groupby(['date', 'hr']).first().reset_index()
df_API['ts'] = df_API["ts"].dt.round('h')  #Round the datestamp column to hours
df_API = df_API.drop(["hr", "date"], axis='columns')

,ts,Temperature_API
859,2024-10-03 06:00:00,6.9
860,2024-10-03 07:00:00,6.5
861,2024-10-03 08:00:00,6.1
862,2024-10-03 09:00:00,6.7
863,2024-10-03 10:00:00,8.7


### Export json to entity dataset in DataFoundry (last5days)

In [15]:
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_window,
    df_indoor,
    df_door,
    df_API
]
# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))
# Use reduce to merge all DataFrames in one go
df_merge = reduce(merge_asof, dataframes_to_merge)
# get a column with only the day and month (lvgl labeling purposes)
df_merge['date'] = df_merge['ts'].dt.strftime('%d/%m') 


,ts,curtain_left,shade_left,distance_right,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date
295,2024-10-03 08:00:00,1.0,1.0,802.0,1,1,0.0,18.5,1.0,6.1,03/10
296,2024-10-03 09:00:00,1.0,1.0,802.0,1,1,0.0,18.7,0.0,6.7,03/10
297,2024-10-03 10:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10
298,2024-10-03 11:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10
299,2024-10-03 12:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10


In [17]:
## get difference between today midnight and the timestamp for each row
today = datetime.now()
today_morning = today.strftime('%Y-%m-%d') + "T00:00:00"
df_merge["time_since_today"] = today_morning
df_merge['time_since_today'] = pd.to_datetime(df_merge['time_since_today']) ## Turn timestamp into datetime dtype
df_merge['time_since_today'] = (df_merge.ts - df_merge.time_since_today) / pd.Timedelta(hours=1)
df_merge = df_merge.drop_duplicates(subset=["ts"]) ## Drop duplicate rows
# # get rows for the last three days only, and reset the index
# df_merge = df_merge[(df_merge.time_since_today > -73) & (df_merge.time_since_today < 0)]
# df_merge = df_merge.reset_index(drop=True)

# Get the min and max values for 'Temperature_indoor' and 'Temperature_API' to set the range in the lvgl chart
minvalue = df_merge[["Temperature_indoor", "Temperature_API"]].min().min()  # Overall minimum value
maxvalue = df_merge[["Temperature_indoor", "Temperature_API"]].max().max()  # Overall maximum value

# Assign the min and max values to new columns 'pp1' and 'pp2'
df_merge["pp1"] = minvalue
df_merge["pp2"] = maxvalue

# Convert specified columns to strings, as only strings can be passed as JSON parameters
columns_to_convert = ['pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_merge[columns_to_convert] = df_merge[columns_to_convert].astype('object')
df_merge['ts'] = df_merge['ts'].map(str)
df_merge['date'] = df_merge['date'].astype('object')

df_merge.tail()

,ts,curtain_left,shade_left,distance_right,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date,time_since_today,pp1,pp2
295,2024-10-03 08:00:00,1.0,1.0,802.0,1,1,0.0,18.5,1.0,6.1,03/10,8.0,3.6,26.6
296,2024-10-03 09:00:00,1.0,1.0,802.0,1,1,0.0,18.7,0.0,6.7,03/10,9.0,3.6,26.6
297,2024-10-03 10:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10,10.0,3.6,26.6
298,2024-10-03 11:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10,11.0,3.6,26.6
299,2024-10-03 12:00:00,1.0,0.0,799.0,1,0,0.0,19.5,0.0,8.7,03/10,12.0,3.6,26.6


In [19]:
df_merge = df_merge.reindex(index=df_merge.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
#get a string id per row to pass as unique resource_id
# Reset index and create a unique 'id' column
df_merge.reset_index(inplace=True, drop=True)
df_merge["id"] = df_merge.index + 1
df_merge['id'] = df_merge['id'].map(str)\


# Get rows for the last three days only, and reset the index for each subset
df_yesterday = df_merge[(df_merge.time_since_today > -25) & (df_merge.time_since_today < 0)]
df_yesterday = df_yesterday.reset_index(drop=True)  # Reset index for yesterday's data

df_twodaysago = df_merge[(df_merge.time_since_today > -49) & (df_merge.time_since_today < -24)]
df_twodaysago = df_twodaysago.reset_index(drop=True)  # Reset index for two days ago's data

df_threedaysago = df_merge[(df_merge.time_since_today > -73) & (df_merge.time_since_today < -48)]
df_threedaysago = df_threedaysago.reset_index(drop=True)  # Reset index for three days ago's data

In [21]:
#### Create a df for today
# Get the current time
now = datetime.now()
today_morning = now.strftime('%Y-%m-%d') + "T00:00:00"
today_range = pd.date_range(start=today_morning, periods=24, freq='H') # Create a date range for the last 24 hours with hourly frequency
today_full = pd.DataFrame(today_range, columns=['ts']) # Create the DataFrame

df_today = df_merge[(df_merge.time_since_today >= 0) & (df_merge.time_since_today < 24)].copy() # get the values that have been measured up until the current time today
df_today['ts'] = pd.to_datetime(df_today['ts'])  # Convert 'ts' to datetime dtype

# Create a list of DataFrames to merge
dataframes_to_merge = [
    today_full,
    df_today]
# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('50min'))

# Use reduce to merge all DataFrames in one go
df_today = reduce(merge_asof, dataframes_to_merge)
df_today = df_today.reset_index(drop=True)  # Reset index for the merged data
df_today['ts'] = df_today['ts'].map(str) # make sure ts is a string type again (for json)
df_today = df_today.reindex(index=df_today.index[::-1]) # reverse the dataframe so that the JSON will eventually be sent in the right order
df_today.reset_index(inplace=True, drop=True)
df_today["id"] = df_today.index + 1
df_today['id'] = df_today['id'].map(str)

df_today

,ts,curtain_left,shade_left,distance_right,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date,time_since_today,pp1,pp2,id
0,2024-10-03 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2024-10-03 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
2,2024-10-03 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,2024-10-03 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
4,2024-10-03 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
5,2024-10-03 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
6,2024-10-03 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
7,2024-10-03 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
8,2024-10-03 15:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
9,2024-10-03 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10


In [23]:
def update_nan_times(df):
    """
    Update the 'NaN_times' column in the given DataFrame to indicate the indexes of NaN values
    in specific columns and fill NaN values with 0.
    
    Parameters:
    df (DataFrame): The DataFrame to be processed.
    """
    df["NaN_times"] = "0"  # Create a column to store the notifications for NaN values
    column_index = 0  # Initialize the index for updating the NaN_times column

    # Iterate over the relevant columns to check for NaN values
    for column in df[['window_door', 'curtain_left', 'curtain_right', 'shade_left', 'shade_right', 'door_indoors']]:
        na_values = df[column].isna().tolist()  # Create a list indicating where NaN values are present
        na_values = np.flip(na_values)
        na_values = [i for i, n in enumerate(na_values) if n == True]  # List all indexes where NaN is True
        # Check if there are any NaN values found
        if any(na_values) == True:
            # If NaN values are found, create a string indicating the first and last index of NaNs
            na_values = [na_values[0]] + [na_values[-1]]  # Get the first and last index of NaN values
            # na_values = 'na: ' + '-'.join(str(x) for x in na_values)  # Format the output string
            na_values = ""  # If no NaN values, set to an empty string
        else:
            na_values = ""  # If no NaN values, set to an empty string

        # Update the NaN_times column with the formatted NaN information for the current column
        df.loc[column_index, 'NaN_times'] = na_values  
        column_index += 1  # Increment the index for the next iteration

    # Fill NaN values with 0 (since they will mess up the JSON output)
    df.fillna("0", inplace=True)

# Now apply the function to the three DataFrames
update_nan_times(df_yesterday)
update_nan_times(df_twodaysago)
update_nan_times(df_threedaysago)
update_nan_times(df_today)

# Make sure again that all columns are of object dtype
columns_to_convert = ['ts', 'date','pp1', 'pp2', 'curtain_left', 'door_indoors', 
                      'shade_left', 'window_door', 'curtain_right', 'shade_right', 
                      'time_since_today', 'Temperature_API', 'Temperature_indoor']
df_yesterday[columns_to_convert] = df_yesterday[columns_to_convert].astype('object')
df_twodaysago[columns_to_convert] = df_twodaysago[columns_to_convert].astype('object')
df_threedaysago[columns_to_convert] = df_threedaysago[columns_to_convert].astype('object')
df_today[columns_to_convert] = df_today[columns_to_convert].astype('object')
df_today

C:\Users\20204113\AppData\Local\Temp\ipykernel_38480\2490485722.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("0", inplace=True)


,ts,curtain_left,shade_left,distance_right,curtain_right,shade_right,window_door,Temperature_indoor,door_indoors,Temperature_API,date,time_since_today,pp1,pp2,id,NaN_times
0,2024-10-03 23:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,1,
1,2024-10-03 22:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,2,
2,2024-10-03 21:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,3,
3,2024-10-03 20:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,4,
4,2024-10-03 19:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,5,
5,2024-10-03 18:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,6,
6,2024-10-03 17:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,7,0
7,2024-10-03 16:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,8,0
8,2024-10-03 15:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,9,0
9,2024-10-03 14:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,10,0


In [25]:
def post_data_to_api(df, url, api_token):
    """
    Function to post data to the specified API endpoint using the provided dataframe.
    
    Parameters:
    df (DataFrame): The dataframe containing the data to be sent.
    url (str): The API endpoint URL.
    api_token (str): The API token for authentication.
    """
    # Loop through the dataframe to add rows to the DF dataset as JSON
    for ind in df.index:
        # Create a dictionary for headers to be sent to the API
        HEADERS = {
            'api_token': api_token,  # API token for authentication
            'resource_id': df["id"].iloc[ind],  # Resource ID from the dataframe
            'token': 'token_for_identifier'  # Token for additional authentication
        }
        
        # Extract parameters from the dataframe for the current index
        ts = df['ts'].iloc[ind]
        temp_out = df['Temperature_API'].iloc[ind]
        temp_in = df['Temperature_indoor'].iloc[ind]
        timesince = df['time_since_today'].iloc[ind]
        window_door = df['window_door'].iloc[ind]
        curtain_left = df['curtain_left'].iloc[ind]
        shade_left = df['shade_left'].iloc[ind]
        curtain_right = df['curtain_right'].iloc[ind]
        shade_right = df['shade_right'].iloc[ind]
        pp1 = df['pp1'].iloc[ind]
        pp2 = df['pp2'].iloc[ind]
        door_indoors = df['door_indoors'].iloc[ind]
        date_yesterday = df_yesterday['date'].iloc[ind]
        date_twodaysago = df_twodaysago['date'].iloc[0]
        date_threedaysago = df_threedaysago['date'].iloc[0]
        NaN_times = df['NaN_times'].iloc[ind]
        
        # Prepare the parameters to be sent in the POST request
        PARAMS = {
            "pp1": pp1, 
            "pp2": pp2, 
            "door_indoors": door_indoors, 
            "window_door": window_door, 
            "curtain_left": curtain_left, 
            "shade_left": shade_left, 
            "curtain_right": curtain_right, 
            "shade_right": shade_right, 
            "Temperature_outdoor": temp_out, 
            "Temperature_indoor": temp_in, 
            "time_since_today": timesince, 
            "ts": ts,
            "date_yesterday": date_yesterday,
            "date_twodaysago": date_twodaysago,
            "date_threedaysago": date_threedaysago,
            "NaN_times": NaN_times
        }    
        
        # Send a POST request to the API with the headers and parameters
        r = requests.post(url=url, headers=HEADERS, json=PARAMS)

url_twodaysago = "https://data.id.tue.nl/datasets/entity/11744/item/"
api_token_twodaysago = "NlBRYlJVWlRDS09LbHlDUlJpNXVqbjVuV1ZuV0hQSmNlemoxQkZsdXFBRT0="  

url_threedaysago = "https://data.id.tue.nl/datasets/entity/11746/item/"
api_token_threedaysago = "d1QwS3ZWSjBnR3U1RVREK3JtOEZLa0hkMitEY25GV3FBM3VkaEZ5Rm9uaz0="  

url_yesterday = "https://data.id.tue.nl/datasets/entity/11745/item/"
api_token_yesterday = "UzBNRUlmcE13ckJmRS9aSGxDdCszaXh2YTdrL0Q2QWhiUTFNMWhrd3g5bz0="    

url_today = "https://data.id.tue.nl/datasets/entity/11930/item/"
api_token_today = "V2JLS3FGamhwZDRCS3ZSVU96SlJXYVIxM3hhMVZORnNlSlMvRmVyZUFsdz0=" 

# Call the function for each dataframe
post_data_to_api(df_yesterday, url_yesterday, api_token_yesterday)
post_data_to_api(df_twodaysago, url_twodaysago, api_token_twodaysago)
post_data_to_api(df_threedaysago, url_threedaysago, api_token_threedaysago)
post_data_to_api(df_today, url_today, api_token_today)